# Layout access benchmark: dense pack vs full sphere

Empirical question: with current xarray + xdggs, does opening a full-sphere zagg-style zarr store materialize the cell_ids coordinate? Does `.sel()` on a small region work cheaply when 99% of the chunk grid is unpopulated? At what child_order (if any) does the full-sphere layout become painful?

**Inputs:** stores produced by `bench/layout_materialize.py` (dense + fullsphere at orders 8, 10, 12).

**Method:** measure RSS deltas and wall-clock at each step (open, decode-with-xdggs, .sel small subset, .values on coord, full-array mean reduction). Compare layouts side by side.

**Required versions** (pin in your env):
- `xarray >= 2024.10` (flexible-indexes API stable)
- `xdggs >= 0.2`     (DGGSIndex with HEALPix)
- `zarr >= 3.1`      (v3 API)

Install with: `uv pip install -e ".[analysis]"` (or pin tighter if you want reproducibility).

In [1]:
import gc
import os
import time
from dataclasses import dataclass, field

import numpy as np
import psutil
import xarray as xr
import xdggs  # noqa: F401  -- registers DGGSIndex on import
import zarr

print(f"xarray  {xr.__version__}")
print(f"xdggs   {xdggs.__version__}")
print(f"zarr    {zarr.__version__}")

xarray  2026.4.0
xdggs   0.6.0
zarr    3.1.5


In [2]:
# Point this at the --base used by layout_materialize.py.
BASE = "s3://xagg/bench-layout"
ORDERS = [8, 10, 12]        # whichever orders you materialized
PARENT_OFFSET = 6           # must match the materialize script
S3_REGION = "us-west-2"

def store_paths(base, orders, parent_offset=6):
    paths = []
    for c in orders:
        p = max(0, c - parent_offset)
        for layout, prefix in (("dense", "dense"), ("fullsphere", "full")):
            paths.append((layout, p, c, f"{base.rstrip('/')}/{prefix}_p{p}_c{c}.zarr"))
    return paths

PATHS = store_paths(BASE, ORDERS, PARENT_OFFSET)
for layout, p, c, path in PATHS:
    print(f"{layout:11s} p={p} c={c}  {path}")

dense       p=2 c=8  s3://xagg/bench-layout/dense_p2_c8.zarr
fullsphere  p=2 c=8  s3://xagg/bench-layout/full_p2_c8.zarr
dense       p=4 c=10  s3://xagg/bench-layout/dense_p4_c10.zarr
fullsphere  p=4 c=10  s3://xagg/bench-layout/full_p4_c10.zarr
dense       p=6 c=12  s3://xagg/bench-layout/dense_p6_c12.zarr
fullsphere  p=6 c=12  s3://xagg/bench-layout/full_p6_c12.zarr


In [3]:
PROC = psutil.Process(os.getpid())

def rss_mb():
    return PROC.memory_info().rss / (1024 * 1024)

def step(label, fn):
    """Run fn(), return (label, wall_s, rss_delta_mb, return_value)."""
    gc.collect()
    rss0 = rss_mb()
    t0 = time.perf_counter()
    val = fn()
    dt = time.perf_counter() - t0
    drss = rss_mb() - rss0
    return {"step": label, "wall_s": dt, "rss_delta_mb": drss, "value": val}

In [4]:
def _open_store(path):
    """Return a zarr Store for either local path or s3://."""
    if path.startswith("s3://"):
        from zagg.store import open_store
        return open_store(path, read_only=True)
    return path  # xarray accepts local paths directly

def _open_zarr(store, group):
    """Open with dask explicitly (avoid cubed-xarray taking the slot)."""
    try:
        return xr.open_zarr(store, group=group, consolidated=True,
                            decode_cf=False, chunks={},
                            chunked_array_type="dask")
    except (TypeError, ValueError):
        return xr.open_zarr(store, group=group, consolidated=True,
                            decode_cf=False, chunks={})

def benchmark_one(layout, parent_order, child_order, path):
    """Run the access pattern suite against one materialized store."""
    metrics = []
    
    # 1) Open with xarray + dask
    def _open_raw():
        return _open_zarr(_open_store(path), str(child_order))
    metrics.append(step("open_zarr", _open_raw))
    ds = metrics[-1]["value"]
    
    # 2) Decode with xdggs index. zagg writes the zarr-conventions/dggs
    #    v1 spec, so we pass convention="zarr". name="cell_ids" is also
    #    required due to an upstream xdggs bug (Zarr.decode uses `name`
    #    as the index dict key and defaults to None, which then trips
    #    xarray with "TypeError: keywords must be strings").
    def _decode():
        try:
            return xdggs.decode(ds, convention="zarr", name="cell_ids")
        except Exception as e:
            return f"decode_failed: {type(e).__name__}: {e}"
    metrics.append(step("xdggs.decode", _decode))
    
    # 3) Find a small populated subset via cell_ids.isin — forces touching
    #    every chunk's cell_ids. On full sphere most chunks are fill_value
    #    (zero), so the GET I/O should still be ~populated count, but dask
    #    has to enumerate the full chunk grid.
    sample_ids = ds.cell_ids.isel(cells=slice(0, 64)).values
    
    def _sel_small():
        mask = ds.cell_ids.isin(sample_ids).compute()
        return int(mask.sum())
    metrics.append(step("sel_small_64cells_mask", _sel_small))
    
    # 4) Materialize the full cell_ids coord (the .values footgun)
    def _materialize_coord():
        return np.asarray(ds.cell_ids.values)
    metrics.append(step("cell_ids.values", _materialize_coord))
    
    # 5) Reduce h_mean over the whole array
    def _full_mean():
        result = ds.h_mean.mean()
        if hasattr(result, 'compute'):
            result = result.compute()
        return float(result)
    metrics.append(step("h_mean.mean", _full_mean))
    
    return metrics

# Sanity-check on the smallest store first
first = PATHS[0]
print(f"\nDry-run on {first[0]} c={first[2]}:")
for m in benchmark_one(*first):
    val_str = ""
    if isinstance(m["value"], str) and m["value"].startswith("decode_failed"):
        val_str = "  [" + m["value"][:80] + "]"
    print(f"  {m['step']:22s} {m['wall_s']*1000:8.1f} ms  {m['rss_delta_mb']:+7.1f} MB{val_str}")


Dry-run on dense c=8:
  open_zarr                 529.3 ms    +44.5 MB
  xdggs.decode              524.9 ms     +7.0 MB
  sel_small_64cells_mask    387.0 ms     +3.8 MB
  cell_ids.values           396.4 ms     +3.0 MB
  h_mean.mean               509.0 ms     +0.2 MB


In [5]:
# Full sweep across all (layout, order) combinations.
rows = []
for layout, p, c, path in PATHS:
    print(f"\n--- {layout} parent={p} child={c} ---")
    try:
        for m in benchmark_one(layout, p, c, path):
            rows.append({"layout": layout, "parent": p, "child": c,
                         "step": m["step"],
                         "wall_s": m["wall_s"],
                         "rss_delta_mb": m["rss_delta_mb"]})
            print(f"  {m['step']:22s} {m['wall_s']*1000:8.1f} ms  {m['rss_delta_mb']:+7.1f} MB")
    except Exception as e:
        print(f"  FAILED: {e}")
        rows.append({"layout": layout, "parent": p, "child": c,
                     "step": "ERROR", "wall_s": float('nan'),
                     "rss_delta_mb": float('nan'), "error": str(e)})


--- dense parent=2 child=8 ---
  open_zarr                 234.3 ms     +0.1 MB
  xdggs.decode              404.8 ms     +2.4 MB
  sel_small_64cells_mask    404.8 ms     +0.6 MB
  cell_ids.values           319.9 ms     +2.3 MB
  h_mean.mean               369.4 ms     +0.1 MB

--- fullsphere parent=2 child=8 ---
  open_zarr                 216.2 ms     +0.0 MB
  xdggs.decode              856.1 ms    +10.6 MB
  sel_small_64cells_mask    849.5 ms     -1.3 MB
  cell_ids.values           618.4 ms    +10.5 MB
  h_mean.mean               767.5 ms     +0.1 MB

--- dense parent=4 child=10 ---
  open_zarr                 175.2 ms     +0.0 MB
  xdggs.decode             5518.7 ms    +61.0 MB
  sel_small_64cells_mask   4835.0 ms    +13.5 MB
  cell_ids.values          4169.2 ms    +33.0 MB
  h_mean.mean              5446.6 ms     +0.1 MB

--- fullsphere parent=4 child=10 ---
  open_zarr                 211.3 ms     +0.0 MB
  xdggs.decode            10867.7 ms   +166.2 MB
  sel_small_64cells_mask  1

In [7]:
import pandas as pd
df = pd.DataFrame(rows)

# Side-by-side: dense vs fullsphere per step per order
for col in ("wall_s", "rss_delta_mb"):
    print(f"\n=== {col} ===")
    pivot = df.pivot_table(
        index=["child", "step"], columns="layout", values=col,
    )
    pivot["ratio"] = pivot.get("fullsphere", float('nan')) / pivot.get("dense", float('nan'))
    print(pivot.round(3))


=== wall_s ===
layout                        dense  fullsphere   ratio
child step                                             
8     cell_ids.values         0.320       0.618   1.933
      h_mean.mean             0.369       0.768   2.078
      open_zarr               0.234       0.216   0.923
      sel_small_64cells_mask  0.405       0.849   2.099
      xdggs.decode            0.405       0.856   2.115
10    cell_ids.values         4.169       9.484   2.275
      h_mean.mean             5.447      11.341   2.082
      open_zarr               0.175       0.211   1.206
      sel_small_64cells_mask  4.835      11.804   2.441
      xdggs.decode            5.519      10.868   1.969
12    cell_ids.values         5.233     135.469  25.887
      h_mean.mean             6.907     147.474  21.352
      open_zarr               0.227       0.322   1.415
      sel_small_64cells_mask  6.042     168.423  27.874
      xdggs.decode            6.932     136.277  19.660

=== rss_delta_mb ===
layout    

## What to look for

1. **`open_zarr` and `xdggs.decode`** — should be near-constant across orders for both layouts. If full-sphere RSS grows linearly with array shape on open, that means xarray is materializing the coord eagerly and the foot-gun is real. If RSS stays flat, lazy coords are working as advertised.
2. **`sel_small_64cells`** — should be O(chunks_touched), not O(array_size). If full-sphere is dramatically slower than dense at order 12, it means xdggs's index is not pruning the unpopulated chunk grid efficiently.
3. **`cell_ids.values`** — this is the explicit materialization. Expected RSS delta:
   - dense order 12:    ~42 MB
   - full   order 12:   ~1.6 GB
   This step is the price of `.values` on the whole coord; it's deliberate, not implicit. The question is whether the user can avoid triggering it during normal use.
4. **`h_mean.mean`** — full-array reduction. Both layouts should touch the same number of *populated* chunks (only ~1300). Full-sphere should not pay extra for unpopulated chunks because zarr resolves fill_value chunks without I/O. If full-sphere is much slower, something in the dask/xarray graph is iterating the empty chunk grid.

## Decision criteria

- **If `open_zarr` and `sel_small` show no significant penalty for full sphere across orders 8-12:** full-sphere wins on protocol cleanliness; we can recommend it.
- **If `open_zarr` materializes the coord (large RSS delta on open at order 12):** stick with base-cell or dense-pack; full-sphere has a real foot-gun.
- **If `cell_ids.values` is the *only* place full-sphere costs more:** that's a deliberate user action and an acceptable trade. Document it and pick full-sphere.
- **If `h_mean.mean` scales with array size rather than populated-chunk count:** there's an xarray/dask graph-build issue that punishes full-sphere; report upstream and pick dense-pack/base-cell as a workaround.